# 13 · chf3d Phase C — dodecahedral coherent multi-beam focus

Run the 12-driver dodecahedral coherent harmonic focus on a Gemini-class
shot, plot the resulting focal-volume slice, and overlay the closed-form
phase-locking decay law on simulated points to check the Phase C kernel
against the analytic baseline.

In [ ]:
%matplotlib inline
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from harmonyemissions.chf import (
    FocalVolume,
    analytic_phase_lock,
    build_beam_array,
    coherent_sum_homogeneous,
    with_phases,
)
from harmonyemissions.config import LaserArrayConfig, load_config
from harmonyemissions.runner import simulate_from_config

## Run the dodecahedral CHF pipeline

In [ ]:
cfg = load_config(Path('..') / 'configs' / 'chf3d_dodecahedral.yaml')
# Tighten the pipeline grid for notebook responsiveness.
cfg.numerics.pipeline_grid = 48
cfg.numerics.chf_focal_volume_n = 12
result = simulate_from_config(cfg)
print('chf_gain breakdown:')
for k, v in result.chf_gain.items():
    print(f'  {k}: {v:.3e}' if isinstance(v, float) else f'  {k}: {v}')

## Focal-volume z-slice at the central diagnostic harmonic

In [ ]:
vol = result.chf_focal_volume
h_idx = len(vol.coords['harmonic_diag']) // 2
n = vol.shape[1]
z_slice = vol.values[h_idx, :, :, n // 2]
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(z_slice, origin='lower', cmap='inferno')
h_value = int(vol.coords['harmonic_diag'].values[h_idx])
ax.set_title(f'I(x, y, z=0) at harmonic n = {h_value}')
ax.set_xlabel('x voxel')
ax.set_ylabel('y voxel')
plt.colorbar(im, ax=ax, label='|E|² (arb)')
plt.tight_layout()
plt.show()

## Phase-locking decay against the closed-form law

Inject Gaussian-σ phase noise on top of the analytic phase lock and
measure the average focal-peak intensity. Compare against the
closed-form interpolation
$$\langle\text{gain}\rangle \approx N^2 e^{-\sigma^2} + N(1 - e^{-\sigma^2})$$

In [ ]:
beam = build_beam_array(LaserArrayConfig(geometry='dodecahedral'))
N = beam.n_beams
h_value = 15
wavelength_m = 0.8e-6
diag = np.array([h_value])
synthetic = []
for _ in range(N):
    far = np.zeros((1, 8, 8), dtype=complex)
    far[:, 4, 4] = 1.0 + 0.0j
    synthetic.append(far)
phases_lock = analytic_phase_lock(
    beam, np.ones(N, dtype=complex), h_value, wavelength_m,
)
rng = np.random.default_rng(42)
sigmas = np.linspace(0.0, np.pi, 9)
means, predicted = [], []
for sig in sigmas:
    trials = []
    for _ in range(64):
        beam_noisy = with_phases(beam, phases_lock + rng.normal(0.0, sig, size=N))
        acc = coherent_sum_homogeneous(
            beam_noisy, synthetic, diag, wavelength_m,
            FocalVolume(n=1, extent_m=0.0),
        )
        trials.append(float(acc.peak_intensity()[0]))
    means.append(np.mean(trials))
    decay = np.exp(-sig * sig)
    predicted.append(N * N * decay + N * (1.0 - decay))

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(sigmas, predicted, 'k-', label='closed-form  $N^2 e^{-\\sigma^2} + N(1-e^{-\\sigma^2})$')
ax.plot(sigmas, means, 'o', label='Phase C accumulator (64 trials each)')
ax.set_xlabel('phase-locking $\\sigma$ [rad]')
ax.set_ylabel(f'mean focal gain (N = {N})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()